<a href="https://colab.research.google.com/github/Srinath77-del/network-intrusion-detection-ml/blob/main/UNSW_NB15_Week5_Model_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import files

uploaded = files.upload()

Saving UNSW_NB15_testing-set.csv to UNSW_NB15_testing-set (1).csv


In [11]:
import pandas as pd

df = pd.read_csv("UNSW_NB15_testing-set (1).csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

Dataset loaded successfully!
Dataset shape: (82332, 45)


In [12]:
X = df.drop(columns=["id", "attack_cat", "label"])
y = df["label"]

print("Input feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Input feature shape: (82332, 42)
Target shape: (82332,)

Target distribution:
label
1    45332
0    37000
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape, y_train.shape)
print("Testing set:", X_test.shape, y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training set: (65865, 42) (65865,)
Testing set: (16467, 42) (16467,)

Training target distribution:
label
1    36265
0    29600
Name: count, dtype: int64

Testing target distribution:
label
1    9067
0    7400
Name: count, dtype: int64


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Categorical columns:", categorical_features)
print("Preprocessing pipeline created successfully!")

Numerical features: 39
Categorical features: 3
Categorical columns: ['proto', 'service', 'state']
Preprocessing pipeline created successfully!


In [15]:
from sklearn.ensemble import RandomForestClassifier

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

baseline_model.fit(X_train, y_train)

print("Baseline Random Forest trained successfully!")

Baseline Random Forest trained successfully!


In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

baseline_pred = baseline_model.predict(X_test)
baseline_proba = baseline_model.predict_proba(X_test)[:, 1]

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_precision = precision_score(y_test, baseline_pred)
baseline_recall = recall_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred)
baseline_roc_auc = roc_auc_score(y_test, baseline_proba)

print("===== BASELINE RANDOM FOREST =====")
print("Accuracy :", round(baseline_accuracy, 4))
print("Precision:", round(baseline_precision, 4))
print("Recall   :", round(baseline_recall, 4))
print("F1-Score :", round(baseline_f1, 4))
print("ROC-AUC  :", round(baseline_roc_auc, 4))

===== BASELINE RANDOM FOREST =====
Accuracy : 0.9752
Precision: 0.983
Recall   : 0.9718
F1-Score : 0.9774
ROC-AUC  : 0.9971


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.base import clone

X_tune, X_val, y_tune, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

experiments = {
    "Experiment 1": {
        "n_estimators": 100,
        "max_depth": 15,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    "Experiment 2": {
        "n_estimators": 100,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    "Experiment 3": {
        "n_estimators": 150,
        "max_depth": 20,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    "Experiment 4": {
        "n_estimators": 150,
        "max_depth": 25,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    }
}

print("Optimization experiments prepared successfully!")
print("Number of experiments:", len(experiments))

Optimization experiments prepared successfully!
Number of experiments: 4


In [18]:
model_1 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

model_1.fit(X_tune, y_tune)

pred_1 = model_1.predict(X_val)

f1_1 = f1_score(y_val, pred_1)

print("===== EXPERIMENT 1 =====")
print("n_estimators: 100")
print("max_depth: 15")
print("min_samples_split: 2")
print("min_samples_leaf: 1")
print("max_features: sqrt")
print("Validation F1-Score:", round(f1_1, 4))

===== EXPERIMENT 1 =====
n_estimators: 100
max_depth: 15
min_samples_split: 2
min_samples_leaf: 1
max_features: sqrt
Validation F1-Score: 0.9707


In [19]:
model_2 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

model_2.fit(X_tune, y_tune)

pred_2 = model_2.predict(X_val)

f1_2 = f1_score(y_val, pred_2)

print("===== EXPERIMENT 2 =====")
print("n_estimators: 100")
print("max_depth: 20")
print("min_samples_split: 2")
print("min_samples_leaf: 1")
print("max_features: sqrt")
print("Validation F1-Score:", round(f1_2, 4))

===== EXPERIMENT 2 =====
n_estimators: 100
max_depth: 20
min_samples_split: 2
min_samples_leaf: 1
max_features: sqrt
Validation F1-Score: 0.9788


In [20]:
model_3 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        max_depth=20,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

model_3.fit(X_tune, y_tune)

pred_3 = model_3.predict(X_val)

f1_3 = f1_score(y_val, pred_3)

print("===== EXPERIMENT 3 =====")
print("n_estimators: 150")
print("max_depth: 20")
print("min_samples_split: 2")
print("min_samples_leaf: 1")
print("max_features: sqrt")
print("Validation F1-Score:", round(f1_3, 4))

===== EXPERIMENT 3 =====
n_estimators: 150
max_depth: 20
min_samples_split: 2
min_samples_leaf: 1
max_features: sqrt
Validation F1-Score: 0.979


In [21]:
model_4 = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        max_depth=25,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

model_4.fit(X_tune, y_tune)

pred_4 = model_4.predict(X_val)

f1_4 = f1_score(y_val, pred_4)

print("===== EXPERIMENT 4 =====")
print("n_estimators: 150")
print("max_depth: 25")
print("min_samples_split: 2")
print("min_samples_leaf: 2")
print("max_features: sqrt")
print("Validation F1-Score:", round(f1_4, 4))

===== EXPERIMENT 4 =====
n_estimators: 150
max_depth: 25
min_samples_split: 2
min_samples_leaf: 2
max_features: sqrt
Validation F1-Score: 0.978
